# KYC Document and Form Routing

A complete TensorFlow Keras example with **1,000 distinct synthetic document images**. Four classes, 250 images each. Each image represents a separate synthetic document; no augmented copies cross the split.

**Task:** Route incoming KYC documents to the correct processing queue: identity card, address proof, income proof, or application form.

This is image classification, not OCR, signature verification, forgery detection, or legal KYC approval. Generic layouts and fictional details are used. Synthetic test results do not estimate performance on real scans.

## Step 1 — Prepare Google Colab
Open this notebook in Colab and run cells from top to bottom. The image ZIP is downloaded automatically from your supplied Google Drive link. Upload **kyc_labels.csv** when prompted, or upload it in the Colab Files sidebar before running.

The Drive ZIP must be accessible to **Anyone with the link — Viewer** for this direct download. If it is restricted or Drive blocks downloads, the download cell explains the error. No Drive mounting is required.


In [ ]:
%pip install -q gdown pandas numpy matplotlib scikit-learn pillow


### Download and extract the image ZIP
The cell checks that the download is a real ZIP before extracting it. A valid existing ZIP is reused when the cell is rerun.

In [ ]:
import os
import zipfile
import gdown

# Download this use case's ZIP from Google Drive.
zip_path = "/content/kyc_images.zip"
extract_folder = "/content/kyc_download"
os.makedirs(extract_folder, exist_ok=True)

if not os.path.isfile(zip_path) or not zipfile.is_zipfile(zip_path):
    try:
        gdown.download(id="1UnlP3vdu29GEjMJJCGQbPwbOX36ax7Dk", output=zip_path, quiet=False)
    except Exception as error:
        raise RuntimeError(
            "Cannot download the Google Drive ZIP. Check sharing: "
            "Anyone with the link / Viewer. Drive download quotas can also block access."
        ) from error

if not zipfile.is_zipfile(zip_path):
    raise RuntimeError("The downloaded file is not a valid ZIP. Check Drive sharing and download access.")

with zipfile.ZipFile(zip_path, "r") as zip_file:
    # Keep all extracted entries inside this use case's folder.
    for member in zip_file.infolist():
        destination = os.path.abspath(os.path.join(extract_folder, member.filename))
        if os.path.commonpath([os.path.abspath(extract_folder), destination]) != os.path.abspath(extract_folder):
            raise ValueError("Unsafe path in ZIP: " + member.filename)
    zip_file.extractall(extract_folder)

image_count = 0
for folder, subfolders, filenames in os.walk(extract_folder):
    for filename in filenames:
        if filename.lower().endswith((".png", ".jpg", ".jpeg")):
            image_count += 1
print("ZIP extracted to:", extract_folder)
print("Image files found (including any previews):", image_count)


### Upload kyc_labels.csv manually
Select the matching CSV when the upload button appears. If it is already uploaded to `/content`, the cell uses that copy.

In [ ]:
from google.colab import files

csv_path = "/content/kyc_labels.csv"
if not os.path.isfile(csv_path):
    print("Please select kyc_labels.csv from your computer.")
    uploaded = files.upload()
    if "kyc_labels.csv" not in uploaded:
        raise FileNotFoundError("Upload the file named kyc_labels.csv, then rerun this cell.")
    with open(csv_path, "wb") as file:
        file.write(uploaded["kyc_labels.csv"])
print("Labels CSV ready:", csv_path)


## Step 2 — Import libraries and fix the random seed
Keras is accessed through `tensorflow.keras`. A seed makes the split and initialization repeatable, although hardware can still affect results.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

keras.utils.set_random_seed(42)
print("TensorFlow:", tf.__version__)
print("Working folder:", os.getcwd())

## Step 3 — Read the uploaded CSV and locate the extracted images
Read **kyc_labels.csv** from `/content`. The notebook locates the matching image folder inside the ZIP, whether the archive contains `images/...` directly or an extra parent folder. It converts the CSV image paths to absolute paths, so later loading and prediction work without changing directories.

`filename` is the image path; `label` is the target class; `split` assigns train/validation/test data. The manually uploaded CSV is used even if the ZIP includes its own labels file.


In [ ]:
df = pd.read_csv("/content/kyc_labels.csv")
required_columns = {"filename", "label", "split", "document_id"}
if not required_columns.issubset(df.columns):
    raise ValueError("CSV must contain filename, label, split and document_id columns.")
if df.empty:
    raise ValueError("The labels CSV is empty.")

# CSV paths look like images/class_name/0000.png.
relative_paths = df["filename"].astype(str).str.replace("\\", "/", regex=False)
first_filename = relative_paths.iloc[0]
image_root = None
for folder, subfolders, filenames in os.walk(extract_folder):
    if os.path.isfile(os.path.join(folder, first_filename)):
        image_root = folder
        break
if image_root is None:
    raise FileNotFoundError("Cannot locate " + first_filename + " inside the extracted ZIP. Check that the CSV matches this ZIP.")

df["filename"] = relative_paths.map(lambda filename: os.path.join(image_root, filename))
print("Image root:", image_root)
class_names = ['identity_card', 'address_proof', 'income_proof', 'application_form']
label_to_id = dict(zip(class_names, range(len(class_names))))
df["label_id"] = df["label"].map(label_to_id)
assert len(df) == 1000
assert df["filename"].nunique() == 1000
assert df["label_id"].notna().all()
for filename in df["filename"]:
    assert os.path.isfile(filename), filename
print(pd.crosstab(df["label"], df["split"]))
df.head()

## Step 4 — Display sample documents
Look at the image together with its expected class. For cheques, inspect the date box, amount box and signature area. For KYC, inspect the document layout and field arrangement.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for index, label in enumerate(class_names):
    samples = df[df["label"] == label].sample(2, random_state=42)
    for row_index, (_, row) in enumerate(samples.iterrows()):
        image = keras.utils.load_img(row["filename"])
        axes[row_index, index].imshow(image)
        axes[row_index, index].set_title(label)
        axes[row_index, index].axis("off")
plt.tight_layout()
plt.show()

## Step 5 — Keep train, validation and test separate
700 training images update the weights. 150 validation images guide early stopping. 150 untouched test images measure the final model. All variants of a real document must stay in one split when adapting this example.

In [ ]:
train_df = df[df["split"] == "train"].copy()
val_df = df[df["split"] == "validation"].copy()
test_df = df[df["split"] == "test"].copy()
assert (len(train_df), len(val_df), len(test_df)) == (700, 150, 150)
assert set(train_df.document_id).isdisjoint(test_df.document_id)
print(len(train_df), len(val_df), len(test_df))

## Step 6 — Load and resize images in batches
Original images are 512 × 768 pixels. Resize to 128 × 192 while preserving their aspect ratio. Each RGB pixel has three channels. `tf.data` reads paths and batches 32 images; labels remain in the same order. Only training data are shuffled. Normalization occurs inside the model.

In [ ]:
IMG_HEIGHT = 128
IMG_WIDTH = 192
BATCH_SIZE = 32

def load_image(filename, label):
    image_bytes = tf.io.read_file(filename)
    image = tf.io.decode_png(image_bytes, channels=3)
    image = tf.image.resize(image, [IMG_HEIGHT, IMG_WIDTH])
    return image, label

def make_dataset(frame, training=False):
    filenames = frame["filename"].astype(str).to_numpy()
    labels_array = frame["label_id"].to_numpy(dtype=np.int32)
    dataset = tf.data.Dataset.from_tensor_slices((filenames, labels_array))
    dataset = dataset.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        dataset = dataset.shuffle(len(frame), seed=42)
    return dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df)
for images, targets in train_ds.take(1):
    print("Image batch:", images.shape)
    print("Label batch:", targets.shape)

## Step 7 — Build four convolution and four pooling layers
A convolution filter is a learned 3 × 3 matrix that finds visual patterns. ReLU introduces nonlinearity. `padding="same"` keeps the spatial dimensions during convolution. Each 2 × 2 max pooling layer keeps the strongest response and halves height and width. Filters start from initialized weights and are learned during training.

| Block | Filters | After convolution | After pooling |
|---|---:|---|---|
| 1 | 16 | 128 × 192 × 16 | 64 × 96 × 16 |
| 2 | 32 | 64 × 96 × 32 | 32 × 48 × 32 |
| 3 | 64 | 32 × 48 × 64 | 16 × 24 × 64 |
| 4 | 128 | 16 × 24 × 128 | 8 × 12 × 128 |

Flatten retains spatial positions, which matter for document fields. Dense combines the learned features. Dropout helps reduce overfitting. Softmax returns four class probabilities. Small rotation and contrast augmentation operate only during training; no horizontal flipping is used for documents.

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
    layers.Rescaling(1.0 / 255),
    layers.RandomRotation(0.01, fill_mode="constant", fill_value=1.0),
    layers.RandomContrast(0.1),
    layers.Conv2D(16, (3, 3), activation="relu", padding="same", name="conv_1"),
    layers.MaxPooling2D((2, 2), name="pool_1"),
    layers.Conv2D(32, (3, 3), activation="relu", padding="same", name="conv_2"),
    layers.MaxPooling2D((2, 2), name="pool_2"),
    layers.Conv2D(64, (3, 3), activation="relu", padding="same", name="conv_3"),
    layers.MaxPooling2D((2, 2), name="pool_3"),
    layers.Conv2D(128, (3, 3), activation="relu", padding="same", name="conv_4"),
    layers.MaxPooling2D((2, 2), name="pool_4"),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(len(class_names), activation="softmax")
], name="document_cnn")
model.summary()
assert sum(isinstance(layer, layers.Conv2D) for layer in model.layers) == 4
assert sum(isinstance(layer, layers.MaxPooling2D) for layer in model.layers) == 4

## Step 8 — Compile the model
Adam updates weights using gradients. Sparse categorical cross entropy compares the four probabilities with the integer class ID. Accuracy counts correct predictions. These are mutually exclusive classes; documents missing multiple cheque fields are outside the current label scheme.

In [ ]:
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

## Step 9 — Train the CNN
An epoch is one pass through the training set. A batch is a smaller group processed together. Backpropagation calculates how each weight contributed to the loss, and Adam adjusts it. Early stopping restores the best validation-loss weights; the test set is not used for training decisions. Training can take several minutes on a CPU.

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2),
]
history = model.fit(train_ds, validation_data=val_ds, epochs=20, callbacks=callbacks)

## Step 10 — Plot learning curves
Falling training loss with rising validation loss suggests overfitting. Training and validation accuracy should be interpreted together. No target accuracy is promised for this dataset.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["accuracy"], label="Train")
axes[0].plot(history.history["val_accuracy"], label="Validation")
axes[0].set_title("Accuracy")
axes[1].plot(history.history["loss"], label="Train")
axes[1].plot(history.history["val_loss"], label="Validation")
axes[1].set_title("Loss")
for axis in axes:
    axis.set_xlabel("Epoch index")
    axis.legend()
plt.tight_layout()
plt.show()

## Step 11 — Evaluate on the held-out test set
Precision asks how often a predicted class is correct. Recall asks how many true examples of that class were found. F1 balances precision and recall. The confusion matrix shows the actual class on rows and predicted class on columns.

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds, verbose=0)
print("Test loss:", round(test_loss, 4))
print("Test accuracy:", round(test_accuracy, 4))
probabilities = model.predict(test_ds, verbose=0)
y_pred = np.argmax(probabilities, axis=1)
y_true = test_df["label_id"].to_numpy()
print(classification_report(y_true, y_pred, labels=range(4), target_names=class_names, zero_division=0))
cm = confusion_matrix(y_true, y_pred, labels=range(4))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(xticks_rotation=30)
plt.tight_layout()
plt.show()

## Step 12 — Display images with actual and predicted labels
The reported score is a softmax probability, not calibrated certainty. Check incorrect predictions visually to understand the failure.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for index, axis in enumerate(axes.flat):
    row = test_df.iloc[index]
    axis.imshow(keras.utils.load_img(row["filename"]))
    predicted_label = class_names[int(y_pred[index])]
    axis.set_title("Actual: " + row["label"] + "\nPredicted: " + predicted_label)
    axis.axis("off")
plt.tight_layout()
plt.show()

## Step 13 — Predict one supplied image
Change `image_path` to your PNG or JPG path. The input image is displayed before its prediction. A 0.70 threshold demonstrates routing low-score results to manual review; it must be validated and calibrated before practical use. A confident score does not detect unfamiliar documents.

In [ ]:
def predict_document(image_path, trained_model):
    original = keras.utils.load_img(image_path)
    resized = keras.utils.load_img(image_path, target_size=(IMG_HEIGHT, IMG_WIDTH))
    image_array = keras.utils.img_to_array(resized)
    batch = np.expand_dims(image_array, axis=0)
    scores = trained_model.predict(batch, verbose=0)[0]
    predicted_id = int(np.argmax(scores))
    plt.figure(figsize=(9, 5))
    plt.imshow(original)
    plt.axis("off")
    plt.show()
    print("Predicted class:", class_names[predicted_id])
    print("Softmax score:", round(float(scores[predicted_id]), 4))
    print("Review required:", bool(scores[predicted_id] < 0.70))
    return pd.DataFrame({"class": class_names, "probability": scores})

image_path = test_df.iloc[0]["filename"]
predict_document(image_path, model)

## Step 14 — Save and reload the model
The `.keras` model includes preprocessing and learned weights. Save class names in the same order as the output neurons. Reloading does not require retraining.

In [ ]:
model.save("document_cnn.keras")
with open("class_names.json", "w") as file:
    json.dump(class_names, file)
reloaded_model = keras.models.load_model("document_cnn.keras")
with open("class_names.json") as file:
    class_names = json.load(file)
predict_document(image_path, reloaded_model)

## Step 15 — Inspect learned feature maps
Feature maps show responses produced by the first convolution layer. They are learned visual features, not extracted text or a guaranteed explanation of a prediction.

In [ ]:
feature_model = keras.Model(inputs=model.inputs, outputs=model.get_layer("conv_1").output)
sample = keras.utils.load_img(image_path, target_size=(IMG_HEIGHT, IMG_WIDTH))
sample_batch = np.expand_dims(keras.utils.img_to_array(sample), axis=0)
feature_maps = feature_model(sample_batch, training=False).numpy()[0]
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for index, axis in enumerate(axes.flat):
    axis.imshow(feature_maps[:, :, index], cmap="viridis")
    axis.set_title("Filter " + str(index + 1))
    axis.axis("off")
plt.tight_layout()
plt.show()

## What to try next
Train with real consented document scans from multiple devices and layouts. Split by customer/document and hold out unseen templates. Include blur, shadows, skew and incomplete crops. Add an unknown-document class or out-of-distribution checks. For multiple simultaneous missing fields, use a multilabel output. For reading names, amounts or dates, add OCR with field-level evaluation. Retain human review for actual cheque/KYC decisions.

References: [TensorFlow image classification](https://www.tensorflow.org/tutorials/images/classification) and [TensorFlow CNN tutorial](https://www.tensorflow.org/tutorials/images/cnn).